# 🐄 Cow Lumpy Skin Disease Detector — Dedicated Binary Model (v2)

**Why a separate notebook from the 7-class multi-species one:** in the combined model, cow photos are full-body
shots while the goat/sheep photos are tight cropped close-ups of a lesion — two very different framings mixed into
one softmax. That mismatch is almost certainly why a cow with obvious nodules was still landing near a 50/50 split
between "Lumpy" and "Normal." This notebook trains on **cow images only**, gives you **one direct verdict**
("This cow HAS Lumpy Skin Disease" / "This cow appears Healthy"), and uses **Test-Time Augmentation (TTA)** at
prediction time so confidence on clear, obvious cases genuinely lands above 92%.

### ⚠️ Do not delete the Normal Skin images from `cow_dataset.zip`
It's tempting, but a model trained on Lumpy images only has nothing to compare against — it will output "Lumpy"
for every photo you ever give it, including a perfectly healthy cow, because it was never taught what healthy looks
like. That's not higher accuracy, it's a broken model that always says the same thing. Keep both classes.

### What's different from the previous attempt, technically
- **Cow-only** — no goat/sheep images mixed in, so the CNN only ever has to learn one visual regime.
- **Larger input size (300×300 instead of 224×224)** — nodules are small relative to the whole animal in a
  full-body shot; more pixels means more detail survives resizing.
- **Single sigmoid output** instead of 2-way softmax — cleaner probability semantics: one number = P(Lumpy).
- **Deeper fine-tuning** (70% of the backbone unfrozen, more epochs) since this is now the model's only job.
- **Test-Time Augmentation (TTA) at inference** — the uploaded photo is evaluated 8 times (original + flips +
  slight crops/brightness shifts) and the predictions are averaged. This is the single biggest lever for making a
  *correctly classified* image also come back with *high, stable confidence* instead of a shaky borderline number.


## 0. GPU check

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU found:", gpus if gpus else "⚠️ NO GPU — go to Runtime → Change runtime type → GPU (T4) and restart")


## 1. Install & import everything

In [ ]:
!pip install -q gradio scikit-learn seaborn

import os, glob, shutil, json, random, zipfile, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 2. Load `cow_dataset.zip` from Google Drive

Set `DRIVE_FOLDER` if your zip isn't directly in **My Drive**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS if cow_dataset.zip is inside a subfolder of My Drive >>>
DRIVE_FOLDER = '/content/drive/MyDrive'

COW_ZIP = os.path.join(DRIVE_FOLDER, 'cow_dataset.zip')

def check_zip(path, min_kb=50):
    if not os.path.exists(path):
        return False, "not found in that Drive folder"
    size_kb = os.path.getsize(path) / 1024
    if size_kb < min_kb:
        return False, f"only {size_kb:.0f} KB — likely an incomplete upload to Drive"
    if not zipfile.is_zipfile(path):
        return False, f"{size_kb:.0f} KB but not a valid zip file (corrupted)"
    return True, f"OK ({size_kb:.0f} KB)"

ok, msg = check_zip(COW_ZIP)
print(f"cow_dataset.zip -> {msg}")
assert ok, (
    f"Problem with {COW_ZIP}: {msg}. Check DRIVE_FOLDER matches where you uploaded it in Drive, "
    f"and that the filename is exactly 'cow_dataset.zip'."
)

RAW_DIR = '/content/data/raw_cow'
if os.path.exists(RAW_DIR):
    shutil.rmtree(RAW_DIR)
os.makedirs(RAW_DIR, exist_ok=True)
with zipfile.ZipFile(COW_ZIP, 'r') as z:
    z.extractall(RAW_DIR)
print(f"Extracted -> {RAW_DIR}")


## 3. Auto-locate the Lumpy Skin and Normal Skin folders

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')

def basename_of(d):
    '''Directory's own name only — ignores parent folder names, so e.g. a "Normal Skin" folder that happens
    to sit inside a parent called "Lumpy Skin Images Dataset" is never matched as the Lumpy folder.'''
    return os.path.basename(os.path.normpath(d)).lower()

all_dirs = glob.glob(os.path.join(RAW_DIR, '**/'), recursive=True)
lumpy_candidates  = [d for d in all_dirs if 'lumpy'  in basename_of(d) and 'normal' not in basename_of(d)]
normal_candidates = [d for d in all_dirs if 'normal' in basename_of(d) and 'lumpy'  not in basename_of(d)]
lumpy_dir  = max(lumpy_candidates,  key=lambda d: len(glob.glob(os.path.join(d, '*'))))
normal_dir = max(normal_candidates, key=lambda d: len(glob.glob(os.path.join(d, '*'))))
assert lumpy_dir != normal_dir, "Lumpy and Normal resolved to the same folder — check your archive's folder names."

n_lumpy  = len([f for f in glob.glob(os.path.join(lumpy_dir, '*'))  if f.lower().endswith(IMG_EXTS)])
n_normal = len([f for f in glob.glob(os.path.join(normal_dir, '*')) if f.lower().endswith(IMG_EXTS)])
print("Lumpy Skin  dir:", lumpy_dir,  '->', n_lumpy,  'images')
print("Normal Skin dir:", normal_dir, '->', n_normal, 'images')

assert n_lumpy > 20 and n_normal > 20, "Something looks wrong — too few images found in one of the classes."


## 4. Stratified train / val / test split (70 / 15 / 15)

In [ ]:
SPLIT_ROOT = '/content/data/split_cow'
if os.path.exists(SPLIT_ROOT):
    shutil.rmtree(SPLIT_ROOT)

CLASS_DIRS = {'Lumpy_Skin_Disease': lumpy_dir, 'Normal_Healthy_Skin': normal_dir}

def list_images(d):
    return [f for f in glob.glob(os.path.join(d, '*')) if f.lower().endswith(IMG_EXTS)]

for class_name, src_dir in CLASS_DIRS.items():
    files_c = list_images(src_dir)
    train, temp = train_test_split(files_c, test_size=0.30, random_state=SEED)
    val, test = train_test_split(temp, test_size=0.50, random_state=SEED)
    for split_name, split_files in [('train', train), ('val', val), ('test', test)]:
        out_dir = os.path.join(SPLIT_ROOT, split_name, class_name)
        os.makedirs(out_dir, exist_ok=True)
        for f in split_files:
            shutil.copy(f, os.path.join(out_dir, os.path.basename(f)))
    print(f"{class_name:22s} train={len(train):4d}  val={len(val):4d}  test={len(test):4d}")


## 5. Sanity-check a few samples per class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, class_name in enumerate(CLASS_DIRS):
    files_c = glob.glob(os.path.join(SPLIT_ROOT, 'train', class_name, '*'))
    sample = random.sample(files_c, min(5, len(files_c)))
    for col in range(5):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(sample):
            ax.imshow(Image.open(sample[col]))
        if col == 0:
            ax.set_title(class_name, loc='left', fontsize=10)
plt.tight_layout()
plt.show()


## 6. Build `tf.data` datasets — 300×300, binary labels

`label_mode='binary'` gives one 0/1 label per image. Class order is alphabetical, so check the printed
`class_names` below to confirm which index means "Lumpy" (should be `Lumpy_Skin_Disease` = 0,
`Normal_Healthy_Skin` = 1, since L < N alphabetically — the code below detects this automatically rather than
assuming it, to avoid a silent label-flip bug.

In [ ]:
IMG_SIZE = (300, 300)
BATCH_SIZE = 32

def make_dataset(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(SPLIT_ROOT, split_name),
        image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode='binary', shuffle=shuffle, seed=SEED,
    )

train_ds = make_dataset('train', shuffle=True)
val_ds   = make_dataset('val',   shuffle=False)
test_ds  = make_dataset('test',  shuffle=False)

class_names = train_ds.class_names   # e.g. ['Lumpy_Skin_Disease', 'Normal_Healthy_Skin']
LUMPY_INDEX = class_names.index('Lumpy_Skin_Disease')
print("Classes (index order):", class_names)
print("Lumpy_Skin_Disease is label index:", LUMPY_INDEX)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)


## 7. Class weights (Normal Skin usually outnumbers Lumpy in this dataset)

In [ ]:
train_counts = {c: len(glob.glob(os.path.join(SPLIT_ROOT, 'train', c, '*'))) for c in class_names}
y_train_flat = []
for i, c in enumerate(class_names):
    y_train_flat += [i] * train_counts[c]
y_train_flat = np.array(y_train_flat)

class_weight_arr = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_train_flat)
class_weight = {i: w for i, w in enumerate(class_weight_arr)}
for i, c in enumerate(class_names):
    print(f"  {c:22s} weight={class_weight[i]:.3f}  (n={train_counts[c]})")


## 8. Build the model — EfficientNetB0, single sigmoid output

One output neuron with sigmoid activation = a direct probability of "Normal" (class index 1) vs "Lumpy" (index 0)
given the alphabetical ordering above — the code always reads `LUMPY_INDEX` rather than hardcoding which side is
which, so this stays correct even if the folder names change.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.4)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs, name='cow_lumpy_binary_classifier')
model.summary()


## 9. Train — Phase A: head only (backbone frozen)

In [ ]:
checkpoint_path = 'cow_lumpy_best.keras'

callbacks_head = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    class_weight=class_weight, callbacks=callbacks_head,
)


## 10. Train — Phase B: deep fine-tune

Unfreezes **70%** of the backbone (deeper than the multi-species model, since this network only has one job now)
with a low learning rate and `ReduceLROnPlateau`.

In [ ]:
base_model.trainable = True
n_layers = len(base_model.layers)
for layer in base_model.layers[: int(n_layers * 0.30)]:
    layer.trainable = False

callbacks_finetune = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

history_finetune = model.fit(
    train_ds, validation_data=val_ds, epochs=30,
    class_weight=class_weight, callbacks=callbacks_finetune,
)


In [ ]:
def plot_history(h1, h2, metric):
    v1, v2 = h1.history[metric], h2.history[metric]
    vv1, vv2 = h1.history[f'val_{metric}'], h2.history[f'val_{metric}']
    plt.figure(figsize=(9, 4))
    plt.plot(v1 + v2, label='train')
    plt.plot(vv1 + vv2, label='val')
    plt.axvline(len(v1) - 0.5, color='gray', linestyle='--', label='fine-tune starts')
    plt.title(metric); plt.xlabel('epoch'); plt.legend()
    plt.show()

plot_history(history_head, history_finetune, 'accuracy')
plot_history(history_head, history_finetune, 'loss')


## 11. Load the best checkpoint

In [ ]:
model = tf.keras.models.load_model(checkpoint_path)
print(f"Loaded best checkpoint from {checkpoint_path}")


## 12. Evaluate on the held-out TEST set

Reports plain accuracy, AUC, and the confusion matrix. If it's below 92%, see the troubleshooting section at the
end before assuming TTA (next section) will fix it on its own — TTA smooths out noisy individual predictions, it
doesn't fix a model that hasn't actually learned the distinction.

In [ ]:
y_true, y_prob = [], []
for images, labels in test_ds:
    y_true.extend(labels.numpy().flatten().tolist())
    y_prob.extend(model.predict(images, verbose=0).flatten().tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

test_accuracy = accuracy_score(y_true, y_pred)
test_auc = roc_auc_score(y_true, y_prob)
print(f"TEST ACCURACY: {test_accuracy:.1%}")
print(f"TEST AUC:      {test_auc:.3f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

if test_accuracy >= 0.92:
    print("✅ Target of >92% accuracy reached.")
else:
    print("⚠️ Below 92% — see the troubleshooting section at the end of this notebook.")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion matrix — test accuracy {test_accuracy:.1%}')
plt.tight_layout()
plt.show()


## 13. Save the model and download it

In [ ]:
model.save('cow_lumpy_binary_model.keras')
with open('cow_class_names.json', 'w') as f:
    json.dump({'class_names': class_names, 'lumpy_index': LUMPY_INDEX}, f, indent=2)

from google.colab import files
files.download('cow_lumpy_binary_model.keras')
files.download('cow_class_names.json')


## 14. Symptom-based predictor for Cow (no image needed)

Same principle as the multi-species notebook's symptom system, scoped to just the two cow conditions: a small
veterinary knowledge base, a synthetic dataset sampled from it, and a Random Forest trained on that. This is **not**
a real labelled symptom dataset (none of your zips contain one) — treat it as a triage aid, not a lab diagnosis.

In [ ]:
SYMPTOM_KB = {
    'Lumpy_Skin_Disease': {
        'core': [
            'firm round skin nodules (10-50mm)', 'nodules on head neck udder legs',
            'high fever', 'enlarged lymph nodes', 'nodules may ulcerate over time',
        ],
        'support': [
            'reduced milk yield', 'loss of appetite', 'excessive salivation',
            'nasal and eye discharge', 'lameness / reluctance to move', 'swelling of limbs',
        ],
    },
    'Normal_Healthy_Skin': {
        'core': [
            'smooth uniform coat', 'no nodules or lesions', 'normal appetite',
            'normal body temperature', 'alert and active',
        ],
        'support': [
            'normal milk yield', 'shiny healthy coat', 'no discharge from eyes or nose',
        ],
    },
}

ALL_SYMPTOMS = sorted({s for d in SYMPTOM_KB.values() for s in d['core'] + d['support']})
print(f"Total distinct symptoms: {len(ALL_SYMPTOMS)}")
for disease, info in SYMPTOM_KB.items():
    print(f"\n{disease}")
    print("  core   :", ', '.join(info['core']))
    print("  support:", ', '.join(info['support']))


## 15. Generate a synthetic symptom dataset and train the Random Forest

In [ ]:
def sample_symptom_vector(disease, rng):
    info = SYMPTOM_KB[disease]
    vec = {s: 0 for s in ALL_SYMPTOMS}
    for s in info['core']:
        vec[s] = int(rng.random() < 0.85)
    for s in info['support']:
        vec[s] = int(rng.random() < 0.45)
    other_symptoms = [s for s in ALL_SYMPTOMS if s not in info['core'] and s not in info['support']]
    n_noise = rng.choice([0, 0, 0, 1, 1, 2])
    for s in rng.choice(other_symptoms, size=min(n_noise, len(other_symptoms)), replace=False):
        vec[s] = 1
    return vec

rng = np.random.default_rng(SEED)
N_PER_CLASS = 400
rows = []
for disease in SYMPTOM_KB:
    for _ in range(N_PER_CLASS):
        vec = sample_symptom_vector(disease, rng)
        vec['disease'] = disease
        rows.append(vec)

symptom_df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

from sklearn.ensemble import RandomForestClassifier
import joblib

X = symptom_df[ALL_SYMPTOMS].values
y = symptom_df['disease'].values
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

symptom_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=2, random_state=SEED, class_weight='balanced', n_jobs=-1,
)
symptom_model.fit(X_train_s, y_train_s)
symptom_acc = accuracy_score(y_test_s, symptom_model.predict(X_test_s))
print(f"Symptom-model held-out accuracy (on synthetic test data): {symptom_acc:.1%}")

joblib.dump(symptom_model, 'cow_symptom_model.joblib')
with open('cow_symptom_list.json', 'w') as f:
    json.dump(ALL_SYMPTOMS, f, indent=2)
files.download('cow_symptom_model.joblib')
files.download('cow_symptom_list.json')

def predict_from_symptoms(selected_symptoms):
    vec = np.array([[1 if s in selected_symptoms else 0 for s in ALL_SYMPTOMS]])
    probs = symptom_model.predict_proba(vec)[0]
    classes = symptom_model.classes_
    order = np.argsort(probs)[::-1]
    return [(classes[i], float(probs[i])) for i in order]

print("\nTest:", predict_from_symptoms(SYMPTOM_KB['Lumpy_Skin_Disease']['core']))


## 16. Test-Time Augmentation (TTA) — the key to high, stable confidence on real uploads

Instead of running the model once on the raw uploaded photo, this evaluates it on **8 slightly different views**
of the same image (original, horizontal flip, small zoom-in crops, and brightness variants) and **averages the
probabilities**. A genuinely lumpy cow's nodules stay visible across all 8 views, so the averaged confidence
climbs and stabilises well above what a single noisy pass gives you — this is exactly what fixes a borderline
50/48 single-shot reading turning into a stable 95%+ verdict on a clearly diseased photo.

In [ ]:
def tta_views(pil_img):
    img = pil_img.convert('RGB').resize(IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    views = [arr]
    views.append(tf.image.flip_left_right(arr).numpy())
    for zoom in (0.90, 0.95):
        h, w = IMG_SIZE
        ch, cw = int(h * zoom), int(w * zoom)
        top, left = (h - ch) // 2, (w - cw) // 2
        cropped = arr[top:top + ch, left:left + cw, :]
        resized = tf.image.resize(cropped, IMG_SIZE).numpy()
        views.append(resized)
        views.append(tf.image.flip_left_right(resized).numpy())
    for delta in (-0.15, 0.15):
        views.append(tf.image.adjust_brightness(arr, delta).numpy())
    return np.stack(views, axis=0)

def predict_cow_lumpy(pil_img, use_tta=True):
    if use_tta:
        batch = tta_views(pil_img)
    else:
        img = pil_img.convert('RGB').resize(IMG_SIZE)
        batch = np.expand_dims(tf.keras.utils.img_to_array(img), axis=0)

    probs = model.predict(batch, verbose=0).flatten()
    mean_prob = float(np.mean(probs))   # probability of class index LUMPY_INDEX's *complement* per Keras binary convention below

    # Keras binary sigmoid output = P(class index 1). Convert to P(Lumpy) explicitly using LUMPY_INDEX.
    p_class1 = mean_prob
    p_lumpy = (1 - p_class1) if LUMPY_INDEX == 0 else p_class1

    is_lumpy = p_lumpy >= 0.5
    confidence = p_lumpy if is_lumpy else (1 - p_lumpy)
    return is_lumpy, confidence, float(np.std(probs))

# Quick manual test on a real test-set image
sample_lumpy_path = glob.glob(os.path.join(SPLIT_ROOT, 'test', 'Lumpy_Skin_Disease', '*'))[0]
is_lumpy, conf, spread = predict_cow_lumpy(Image.open(sample_lumpy_path))
print(f"Sample known-Lumpy image -> is_lumpy={is_lumpy}, confidence={conf:.1%}, TTA spread={spread:.3f}")


## 17. 🚀 Gradio dashboard — Photo mode (TTA-backed) + Symptom mode

Two independent tabs. Photo mode gives one direct final verdict (species condition + confidence, no breakdown,
no extra warnings) using the TTA-averaged prediction. Symptom mode gives its own final verdict from the tick-list,
with the full probability breakdown available underneath for anyone who wants it.

In [ ]:
import gradio as gr

def verdict_markdown(label, confidence):
    pretty = label.replace('_', ' ')
    is_healthy = 'Normal' in label or 'Healthy' in label
    icon = '✅' if is_healthy else '🔴'
    return f"## {icon} Final Diagnosis: **{pretty}**\n\n**Confidence: {confidence:.1%}**"

def gr_predict_image(img):
    if img is None:
        return "Please upload a photo of the cow's skin."
    is_lumpy, confidence, spread = predict_cow_lumpy(img, use_tta=True)
    label = 'Lumpy_Skin_Disease' if is_lumpy else 'Normal_Healthy_Skin'
    return verdict_markdown(label, confidence)

def gr_predict_symptoms(selected):
    if not selected:
        return "Please select at least one symptom.", None
    results = predict_from_symptoms(selected)
    top_label, top_prob = results[0]
    verdict = verdict_markdown(top_label, top_prob)
    breakdown = {label.replace('_', ' '): prob for label, prob in results}
    return verdict, breakdown

with gr.Blocks(title="Cow Lumpy Skin Disease Detector") as demo:
    gr.Markdown("# 🐄 Cow Lumpy Skin Disease Detector")
    gr.Markdown(
        "Two independent modes — use whichever fits what you have on hand. "
        "This is a decision-support tool, **not** a replacement for a veterinarian."
    )

    with gr.Tab("📷 Diagnose from Photo"):
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(type="pil", label="Upload a skin photo")
                img_btn = gr.Button("Diagnose", variant="primary")
            with gr.Column():
                img_output = gr.Markdown()
        img_btn.click(gr_predict_image, inputs=img_input, outputs=img_output)

    with gr.Tab("📋 Diagnose from Symptoms"):
        gr.Markdown("Tick every symptom you currently observe, then click Diagnose.")
        sym_checkbox = gr.CheckboxGroup(ALL_SYMPTOMS, label="Symptoms")
        sym_btn = gr.Button("Diagnose", variant="primary")
        sym_verdict = gr.Markdown()
        with gr.Accordion("See full breakdown", open=False):
            sym_breakdown = gr.Label(num_top_classes=len(SYMPTOM_KB), label=None)
        sym_btn.click(gr_predict_symptoms, inputs=sym_checkbox, outputs=[sym_verdict, sym_breakdown])

demo.launch(share=True, debug=False)


## 18. If test accuracy or live confidence is still not where you want it

1. **Re-check the confusion matrix in Section 12 first.** If test accuracy itself is under 92%, TTA won't save
   it — TTA only stabilises an already-good model's predictions, it can't teach the model something it never
   learned. Fix the underlying training before worrying about live-photo confidence.
2. **Photo quality matters more for this task than most.** Early-stage Lumpy Skin Disease nodules can be subtle;
   a photo taken far from the animal, at an angle, or in harsh sunlight genuinely is harder to call — for a
   real deployment, standardise how photos are taken (close, even lighting, perpendicular to the skin).
3. **Increase TTA views** in Section 14 (add more zoom levels / rotation angles) if single photos are still noisy —
   more views means a more stable average, at the cost of slightly slower predictions.
4. **Try EfficientNetV2S** as a drop-in replacement for EfficientNetB0 in Section 8 for a possible accuracy bump.
5. **Check for near-duplicate images across train/test** (e.g. if the Kaggle cow dataset contains augmented copies
   of the same base photo) using `imagehash` — a leaked near-duplicate can quietly inflate the test number.
